In [2]:
import datetime
import time

import requests

server_url = "http://35.192.21.69:8082"
username = "atifmerghad@gmail.com"
password = "112233"
device_id = 123456789012345


def send_position_osmand(position):
    """
    Send a single position to the Traccar server using the OsmAnd protocol.
    The OsmAnd protocol accepts parameters via query string (or POST) with names like:
      - id (or deviceid)
      - lat, lon
      - timestamp (which can be ISO 8601)
      - speed, accuracy, etc.
    """
    # Build a dictionary of parameters using OsmAnd names
    osmand_params = {
        "id": position["deviceId"],  # mandatory device identifier
        "lat": position["latitude"],  # latitude as a float
        "lon": position["longitude"],  # longitude as a float
        "timestamp": position["fixTime"],  # ISO 8601 timestamp (or epoch)
        "speed": position["speed"],  # speed (units depend on configuration)
        "accuracy": position["accuracy"],  # accuracy in meters
    }

    # You can include other parameters such as bearing, altitude, etc. if needed.
    # For example:
    # osmand_params["bearing"] = position.get("bearing", 0)
    # osmand_params["altitude"] = position.get("altitude", 0)

    # Send the data as a GET request with query parameters
    response = requests.get(
        server_url, params=osmand_params, auth=(username, password)
    )
    print(
        f"Sent position at {position['fixTime']} - Status: {response.status_code}"
    )


def simulate_trip(
    device_id,
    start_coords,
    end_coords,
    num_points,
    start_time,
    interval_seconds,
):
    """
    Simulate a trip by sending a series of positions that interpolate between start_coords and end_coords.
    """
    lat1, lon1 = start_coords
    lat2, lon2 = end_coords
    for i in range(num_points):
        fraction = i / (num_points - 1)
        lat = lat1 + fraction * (lat2 - lat1)
        lon = lon1 + fraction * (lon2 - lon1)
        speed = 50  # km/h (constant speed for simulation)
        # Create an ISO 8601 timestamp (appending 'Z' to indicate UTC)
        fix_time = (
            start_time + datetime.timedelta(seconds=i * interval_seconds)
        ).isoformat() + "Z"
        position = {
            "deviceId": device_id,
            "latitude": lat,
            "longitude": lon,
            "speed": speed,
            "fixTime": fix_time,
            "accuracy": 10,
        }
        send_position_osmand(position)
        # Sleep briefly to avoid sending all positions at once
        time.sleep(0.5)
    return start_time + datetime.timedelta(
        seconds=(num_points - 1) * interval_seconds
    )


def simulate_stop(
    device_id, coords, duration_seconds, start_time, interval_seconds
):
    """
    Simulate a stop by sending several positions with the same coordinates and zero speed.
    """
    num_points = int(duration_seconds // interval_seconds) + 1
    for i in range(num_points):
        fix_time = (
            start_time + datetime.timedelta(seconds=i * interval_seconds)
        ).isoformat() + "Z"
        position = {
            "deviceId": device_id,
            "latitude": coords[0],
            "longitude": coords[1],
            "speed": 0,  # No movement during a stop
            "fixTime": fix_time,
            "accuracy": 5,
        }
        send_position_osmand(position)
        time.sleep(0.5)
    return start_time + datetime.timedelta(
        seconds=(num_points - 1) * interval_seconds
    )


if __name__ == "__main__":
    # Use current UTC time as starting point
    current_time = datetime.datetime.utcnow()

    # Simulate first trip: from Point A to Point B
    pointA = (40.0, -74.0)  # Starting coordinates
    pointB = (40.01, -73.99)  # Ending coordinates for the first trip
    print("Simulating first trip...")
    t1 = simulate_trip(
        device_id,
        pointA,
        pointB,
        num_points=10,
        start_time=current_time,
        interval_seconds=5,
    )

    # Simulate a stop at Point B for 2 minutes (120 seconds)
    print("Simulating stop...")
    t2 = simulate_stop(
        device_id,
        pointB,
        duration_seconds=120,
        start_time=t1,
        interval_seconds=10,
    )

    # Simulate second trip: from Point B to Point C
    pointC = (40.02, -73.98)  # Ending coordinates for the second trip
    print("Simulating second trip...")
    t3 = simulate_trip(
        device_id,
        pointB,
        pointC,
        num_points=10,
        start_time=t2,
        interval_seconds=5,
    )

    print("Simulation complete.")


Simulating first trip...


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))